# 03 · Phân tích kết quả (RQ1–RQ4)

Đọc mọi dự báo trong thư mục kết quả trên Drive, tính độ đo, kiểm định, phân tầng theo loại ngày,
kiểm định tương tác (H2b), mô phỏng tồn kho và diễn giải. Mọi bảng và hình đều sinh từ dự báo đã lưu.

In [ ]:
# ===== THIẾT LẬP CHUNG (chạy đầu tiên mỗi khi mở notebook) =====
GITHUB_USER = "Punpunpuong22"
REPO = "time_series_demand_fc_m5_dataset"
DRIVE_DIR = "/content/drive/MyDrive/M5_project"   # thư mục trên Google Drive để lưu dữ liệu và kết quả

from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull
%cd /content/{REPO}
!pip install -q -e .
!mkdir -p {DRIVE_DIR}/data_raw {DRIVE_DIR}/results

In [ ]:
# ===== DỮ LIỆU M5 =====
# Cách A (khuyến nghị): bạn đã có 3 tệp CSV trong Google Drive tại {DRIVE_DIR}/data_raw
#   calendar.csv, sell_prices.csv, sales_train_evaluation.csv
# Cách B: tải từ Kaggle (bỏ dấu # ở các dòng dưới, cần tệp kaggle.json của bạn)
# from google.colab import files
# files.upload()                       # chọn kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !bash scripts/download_m5.sh {DRIVE_DIR}/data_raw

!mkdir -p data
!ln -sfn {DRIVE_DIR}/data_raw data/raw
!ls -lh data/raw/

In [ ]:
RESULTS = f"{DRIVE_DIR}/results"
!python experiments/evaluate.py --out {RESULTS}

## RQ1 · Độ chính xác tổng thể

In [ ]:
import sys; sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
T = f"{RESULTS}/tables"
overall = pd.read_csv(f"{T}/overall.csv")
display(overall.round(4))
ax = overall.set_index("exp")["wrmsse"].plot(kind="bar", yerr=overall.set_index("exp")["wrmsse_sd"], capsize=3, figsize=(9, 4))
ax.set_ylabel("WRMSSE (trung bình 13 cửa sổ, ± SD qua seed)"); plt.tight_layout(); plt.show()
display(pd.read_csv(f"{T}/dm_tests.csv")); display(pd.read_csv(f"{T}/bootstrap_wrmsse_diff.csv"))

## RQ2 · Đóng góp của đặc trưng lịch cho từng họ mô hình, theo loại ngày (H2a, H2b)
Đóng góp = RMSSE(không lịch) − RMSSE(có lịch); dương nghĩa là lịch giúp giảm sai số.

In [ ]:
bd = pd.read_csv(f"{T}/by_daytype.csv").groupby(["exp", "daytype"])["rmsse_mean"].mean().unstack(0)
contrib = pd.DataFrame({"LightGBM": bd["L-H"] - bd["L-HC"], "LSTM": bd["S-H"] - bd["S-HC"]})
display(contrib.round(4))
contrib.plot(kind="barh", figsize=(8, 5)); plt.axvline(0, c="gray")
plt.xlabel("Giảm RMSSE nhờ đặc trưng lịch"); plt.tight_layout(); plt.show()

### Kiểm định tương tác bằng mô hình hồi quy hỗn hợp

In [ ]:
from m5cal.config import load_config
from m5cal.pipeline import prepare, load_predictions, evaluate
from m5cal.stats_tests import interaction_model
cfg = load_config("configs")
panel, calf, windows = prepare(cfg, out_dir=RESULTS)
preds = load_predictions(RESULTS)
res = evaluate(panel, calf, windows, preds[preds["exp"].isin(["L-H", "L-HC", "S-H", "S-HC"])])
st = res["strat_series"]
st["family"] = np.where(st["exp"].str.startswith("L"), "lgbm", "lstm")
st["feature_set"] = st["exp"].str.split("-").str[1]
st = st.groupby(["sid", "family", "feature_set", "daytype"], as_index=False)["rmsse"].mean()
fit = interaction_model(st)
print(fit.summary())

## RQ3 · Theo nhóm nhu cầu

In [ ]:
display(pd.read_csv(f'{T}/by_demand_class.csv').round(4))

## RQ4 · Tồn kho (mô phỏng, so sánh tương đối)

In [ ]:
inv = pd.read_csv(f"{T}/inventory.csv")
display(inv.pivot(index="daytype", columns="exp", values="fill_rate").round(3))
display(inv.pivot(index="daytype", columns="exp", values="avg_inventory").round(2))

## Diễn giải · permutation importance theo nhóm (thước đo chung) và TreeSHAP (bổ sung)
Huấn luyện lại một mô hình mỗi họ tại mốc tinh chỉnh rồi xáo trộn từng nhóm đặc trưng trên 3 cửa sổ đánh giá đầu.

In [ ]:
from m5cal.pipeline import make_model
from m5cal.explain import grouped_permutation_importance, treeshap_by_group
tune_w = [w for w in windows if w.role == "tune"]
eval_w = [w for w in windows if w.role == "eval"][:3]
imp = {}
for code_, spec in {"L-HCP": {"model": "lgbm", "feature_set": "HCP", "horizon_groups": "weekly"},
                    "S-HCP": {"model": "lstm", "feature_set": "HCP"}}.items():
    m = make_model(cfg, spec, panel, seed=1).fit(panel, calf, tune_w[0].origin, valid_windows=tune_w[:1])
    groups = ["cal_regular", "cal_irregular", "snap", "price"]
    imp[code_] = grouped_permutation_importance(m, panel, calf, eval_w, groups=groups).set_index("group")["wrmsse_increase"]
    if code_ == "L-HCP":
        display(treeshap_by_group(m, panel, calf, eval_w[0]))
display(pd.DataFrame(imp).round(4))